In [1]:
# ============================================================
# ECOSTRESS NIGHTTIME LST PROCESSING PIPELINE
# - Parses APPEEARS ECOSTRESS filenames
# - Applies QC filtering
# - Keeps nighttime overpasses only
# - Aggregates to INSEE 200 m grid using raster-based zonal means
# - Preserves both nighttime-window temperatures
# - Constructs exact raw cooling outcomes once
# - Uses Lambert-93 (EPSG:2154)
#
# >>> THREE-PART REVISION:
#     1. late_night and early_morning remain unmodified monthly medians.
#     2. cooling_magnitude_raw = late_night - early_morning.
#     3. NCD_raw = early_morning - late_night.
#     4. NCD is retained as an exact alias of NCD_raw for compatibility.
#     5. Outlier treatment is stored separately as NCD_winsorized.
#        The raw variables are never clipped or overwritten.
# ============================================================

import geopandas as gpd
import rioxarray as rxr
import rasterio
from rasterio.features import rasterize
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from tqdm import tqdm


In [2]:
# Define paths and parameters
DATA_ROOT = Path("C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/data")
GRID_PATH = "C:/Users/kbonsu/Desktop/AICC Conference/idf_200_grid.shp"
IDF_PATH = "C:/Users/kbonsu/Desktop/AICC Conference/ile-de-france.shp"

GRID_ID_FIELD = "idcar_200m"     # FINAL spatial identifier

OUT_LST = "C:/Users/kbonsu/Desktop/AICC Conference/ecostress_night_LST_200m.csv"

# >>> MODIFIED: output names now refer to the Nighttime Cooling Deficit (NCD)
OUT_NCD = "C:/Users/kbonsu/Desktop/AICC Conference/ecostress_nighttime_cooling_deficit_200m.csv"
OUT_SUMMER_NCD = "C:/Users/kbonsu/Desktop/AICC Conference/ecostress_summer_nighttime_cooling_deficit_200m.csv"


In [3]:
# Ile-de-France boundary 
idf = gpd.read_file(IDF_PATH).to_crs("EPSG:2154")

# INSEE 200 m grid (analysis units)
grid = gpd.read_file(GRID_PATH).to_crs("EPSG:2154")
grid = grid[[GRID_ID_FIELD, "geometry"]].reset_index(drop=True)

# Temporary numeric ID for rasterization only
grid["tmp_id"] = grid.index.astype("int32")
grid_lookup = grid[["tmp_id", GRID_ID_FIELD]].copy()


In [4]:
# Function to parse ECOSTRESS datetime from filename (e.g ECO_L2_LSTE.002_LST_doy2018224162117_aid0001)
# Converts UTC → Europe/Paris local time
def parse_ecostress_datetime(fname):
    part = fname.split("_doy")[1].split("_")[0]
    year = int(part[:4])
    doy  = int(part[4:7])
    time = part[7:13]

    dt_utc = datetime.strptime(
        f"{year} {doy} {time}",
        "%Y %j %H%M%S"
    )

    dt_utc = dt_utc.replace(tzinfo=ZoneInfo("UTC"))
    dt_local = dt_utc.astimezone(ZoneInfo("Europe/Paris"))
    
    return dt_local


In [5]:
# Function to load and preprocess ECOSTRESS LST, cloud mask, and QC layers

def load_ecostress_lst(lst_path, cloud_path, qc_path, idf):

    lst = rxr.open_rasterio(lst_path, masked=True).squeeze()
    cloud = rxr.open_rasterio(cloud_path, masked=True).squeeze()
    qc = rxr.open_rasterio(qc_path, masked=True).squeeze()

    # 1. Clear-sky only (mandatory)
    lst = lst.where(cloud == 0)

    # 2. Remove only invalid QC (do NOT over-filter)
    lst = lst.where(qc != 255)

    # 3. Reproject to Lambert-93
    lst = lst.rio.reproject("EPSG:2154")

    # 4. Clip to EXACT download extent (Île-de-France)
    lst = lst.rio.clip(
        idf.geometry,
        idf.crs,
        drop=True
    )

    # 5. ECOSTRESS L2 LSTE v002 scaling
    # LST (K) = DN × 0.02 → °C
    lst = (lst * 0.02) - 273.15

    # Soft sanity check (warn only)
    vmin = float(lst.min())
    vmax = float(lst.max())
    if vmin < -20 or vmax > 60:
        print(f"⚠️ Warning: wide LST range ({vmin:.1f}, {vmax:.1f})")

    return lst


In [6]:
# Function to rasterize INSEE grid to match ECOSTRESS LST raster (for zonal stats)

def rasterize_grid(grid, ref_raster):
    shapes = (
        (geom, val)
        for geom, val in zip(grid.geometry, grid["tmp_id"])
    )

    return rasterize(
        shapes=shapes,
        out_shape=ref_raster.shape,
        transform=ref_raster.rio.transform(),
        fill=-1,
        dtype="int32"
    )


In [7]:
# Function to compute a raster-based zonal mean LST for each grid cell
# >>> MODIFIED NOTE:
# This is an arithmetic mean of raster-cell values assigned to each INSEE grid cell.
# It is not an explicit fractional-overlap, area-weighted polygon aggregation.
def zonal_mean_fast(lst, grid_raster):
    data = lst.values
    zones = grid_raster

    valid = (zones >= 0) & (~np.isnan(data))

    df = pd.DataFrame({
        "tmp_id": zones[valid],
        "LST": data[valid]
    })

    return df.groupby("tmp_id")["LST"].mean()


In [8]:
# Main processing loop
records = []

for folder in sorted(DATA_ROOT.iterdir()):
    if not folder.is_dir():
        continue

    print(f"\nProcessing folder: {folder.name}")
    lst_files = list(folder.glob("*_LST_doy*.tif"))

    for lst_file in tqdm(lst_files, desc=folder.name):

        cloud_file = lst_file.with_name(
            lst_file.name.replace("_LST_", "_cloud_mask_")
        )
        qc_file = lst_file.with_name(
            lst_file.name.replace("_LST_", "_QC_")
        )

        if not cloud_file.exists() or not qc_file.exists():
            continue

        dt = parse_ecostress_datetime(lst_file.name)

        # Nighttime filter (ECOSTRESS reality)
        if not (dt.hour >= 22 or dt.hour <= 5):
            continue

        lst = load_ecostress_lst(
            lst_file,
            cloud_file,
            qc_file,
            idf
        )

        grid_raster = rasterize_grid(grid, lst)
        means = zonal_mean_fast(lst, grid_raster)

        df = means.reset_index()
        df["datetime"] = dt
        df["date"] = dt.date()
        df["hour"] = dt.hour
        df["year"] = dt.year
        df["month"] = dt.month

        records.append(df)



Processing folder: ECOSTRESS_2018-2019


ECOSTRESS_2018-2019:  81%|████████  | 329/408 [04:22<02:24,  1.83s/it]

⚠️ Warning: wide LST range (-131.2, 74.2)


ECOSTRESS_2018-2019: 100%|██████████| 408/408 [05:36<00:00,  1.21it/s]



Processing folder: ECOSTRESS_2020-2021


...

Olympic Site: 0it [00:00, ?it/s]



Processing folder: summer_ndvi_ndbi_dem


summer_ndvi_ndbi_dem: 0it [00:00, ?it/s]



Processing folder: summer_ndvi_ndbi_dem_2154


summer_ndvi_ndbi_dem_2154: 0it [00:00, ?it/s]


In [9]:
# Merge all records and join back to original grid IDs
ecostress_df = pd.concat(records, ignore_index=True)

ecostress_df = (
    ecostress_df
    .merge(grid_lookup, on="tmp_id", how="left")
    .drop(columns=["tmp_id"])
)



In [10]:
ecostress_df.head() 

,LST,datetime,date,hour,year,month,idcar_200m
0,21.750000,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2805800E3777400
1,22.575005,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2805800E3777600
2,22.654999,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2805800E3777800
3,21.812504,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2806000E3776200
4,22.779236,2018-08-04 22:04:06+02:00,2018-08-04,22,2018,8,CRS3035RES200mN2806000E3777600


In [11]:
ecostress_df.describe()

,LST,hour,year,month
count,1.206942e+07,1.206942e+07,1.206942e+07,1.206942e+07
mean,1.157333e+01,7.272657e+00,2.021604e+03,7.179442e+00
std,6.490305e+00,8.559279e+00,2.126969e+00,2.812938e+00
min,-7.533000e+01,0.000000e+00,2.018000e+03,1.000000e+00
25%,6.562500e+00,2.000000e+00,2.020000e+03,5.000000e+00
50%,1.238733e+01,3.000000e+00,2.022000e+03,7.000000e+00
75%,1.625667e+01,5.000000e+00,2.023000e+03,9.000000e+00
max,3.032700e+02,2.300000e+01,2.025000e+03,1.200000e+01


In [12]:
ecostress_df["LST"].quantile([0.001, 0.01, 0.99, 0.999])

0.001    -4.198569
0.010    -1.603333
0.990    24.342308
0.999    33.056493
Name: LST, dtype: float64

In [13]:
ecostress_df.year.unique()

array([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [14]:
ecostress_df["month"] = ecostress_df["datetime"].dt.month

ecostress_df.groupby("month")["LST"].mean()


month
1      4.091865
2      1.478680
3      2.580386
4     10.105347
5     10.688075
6     17.327547
7     18.327007
8     17.402897
9     14.113669
10     9.899505
11     6.141269
12     2.832694
Name: LST, dtype: float32

In [15]:
# Compute mean LST for each calendar month (across all years) and save results
monthly_mean =  ecostress_df.groupby('month')['LST'].agg(['mean', 'std', 'count']).round(2)

print("Mean LST per month (across all years):")

print(monthly_mean)


monthly_mean.to_csv(
    'C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/ecostress_monthly_mean_LST.csv'
)


Mean LST per month (across all years):
        mean   std    count
month                      
1       4.09  2.15   374265
2       1.48  3.15   329532
3       2.58  2.76  1380077
4      10.11  2.01   105620
5      10.69  2.96  1252755
6      17.33  4.65   372049
7      18.33  3.39  2726903
8      17.40  5.03   207412
9      14.11  2.72  3411091
10      9.90  2.06   203970
11      6.14  2.70  1396500
12      2.83  2.55   309246


In [16]:
ecostress_df.columns

Index(['LST', 'datetime', 'date', 'hour', 'year', 'month', 'idcar_200m'], dtype='str')

In [17]:
# ============================================================
# NIGHTTIME-WINDOW AGGREGATION AND THREE-PART OUTCOME CONSTRUCTION
# ============================================================
# >>> THREE-PART REVISION
#
# Time windows:
#   late night    = 22:00–01:00
#   early morning = 03:00–05:00
#
# The values are monthly medians by grid cell and window, not same-night pairs.
#
# Exact raw identities:
#   cooling_magnitude_raw = late_night - early_morning
#   NCD_raw               = early_morning - late_night
#
# Larger cooling_magnitude_raw = larger temperature decline.
# Higher NCD_raw = weaker cooling.
#
# IMPORTANT:
# Raw variables are never winsorized or overwritten.
# Any outlier-treated version receives a separate variable name.

ecostress_df = ecostress_df.copy()

# Assign nighttime windows
ecostress_df["night_bin"] = pd.NA

# Late night: 22, 23, 0, 1
ecostress_df.loc[
    (ecostress_df["hour"] >= 22) | (ecostress_df["hour"] <= 1),
    "night_bin"
] = "late_night"

# Early morning: 3, 4, 5
ecostress_df.loc[
    (ecostress_df["hour"] >= 3) & (ecostress_df["hour"] <= 5),
    "night_bin"
] = "early_morning"

# Remove the unassigned transition hour(s)
ecostress_df = ecostress_df.dropna(subset=["night_bin"])

# Monthly median and observation count by grid cell and window
grouped = (
    ecostress_df
    .groupby([GRID_ID_FIELD, "year", "month", "night_bin"])["LST"]
    .agg(window_median="median", window_count="count")
    .reset_index()
)

MIN_OBS_PER_WINDOW = 2
grouped = grouped[grouped["window_count"] >= MIN_OBS_PER_WINDOW].copy()

# Window temperatures
night_metrics = (
    grouped
    .pivot(
        index=[GRID_ID_FIELD, "year", "month"],
        columns="night_bin",
        values="window_median"
    )
    .reset_index()
)

# Window-specific counts retained for quality control
night_counts = (
    grouped
    .pivot(
        index=[GRID_ID_FIELD, "year", "month"],
        columns="night_bin",
        values="window_count"
    )
    .reset_index()
    .rename(columns={
        "late_night": "n_late_night",
        "early_morning": "n_early_morning"
    })
)

night_metrics = night_metrics.merge(
    night_counts,
    on=[GRID_ID_FIELD, "year", "month"],
    how="left",
    validate="one_to_one"
)

night_metrics = night_metrics.dropna(
    subset=["late_night", "early_morning"]
).copy()

# Exact raw outcomes
night_metrics["cooling_magnitude_raw"] = (
    night_metrics["late_night"] - night_metrics["early_morning"]
)
night_metrics["NCD_raw"] = (
    night_metrics["early_morning"] - night_metrics["late_night"]
)

# Backward-compatible alias. It must remain identical to NCD_raw.
night_metrics["NCD"] = night_metrics["NCD_raw"]

# Separate winsorized sensitivity variable; raw NCD remains untouched.
q01 = night_metrics["NCD_raw"].quantile(0.01)
q99 = night_metrics["NCD_raw"].quantile(0.99)
night_metrics["NCD_winsorized"] = night_metrics["NCD_raw"].clip(q01, q99)

# Exact identity checks
assert np.allclose(
    night_metrics["NCD_raw"],
    night_metrics["early_morning"] - night_metrics["late_night"],
    equal_nan=True
)
assert np.allclose(
    night_metrics["cooling_magnitude_raw"],
    night_metrics["late_night"] - night_metrics["early_morning"],
    equal_nan=True
)
assert np.allclose(
    night_metrics["NCD_raw"],
    -night_metrics["cooling_magnitude_raw"],
    equal_nan=True
)
assert np.allclose(
    night_metrics["NCD"],
    night_metrics["NCD_raw"],
    equal_nan=True
)

print("Three-part nighttime outcomes created successfully.")
print("NCD_raw is exact and has not been clipped.")
print("NCD_winsorized is available only for sensitivity analysis.")


Three-part nighttime outcomes created successfully.
NCD_raw is exact and has not been clipped.
NCD_winsorized is available only for sensitivity analysis.


In [18]:
# >>> MODIFIED: inspect the revised nighttime metrics
night_metrics.head()


night_bin,idcar_200m,year,month,early_morning,late_night,n_early_morning,n_late_night,cooling_magnitude_raw,NCD_raw,NCD,NCD_winsorized
7,CRS3035RES200mN2805800E3777600,2020,9,9.602505,15.939995,2.0,4.0,6.337490,-6.337490,-6.337490,-6.337490
9,CRS3035RES200mN2805800E3777600,2020,11,3.952499,6.795001,2.0,2.0,2.842502,-2.842502,-2.842502,-2.842502
10,CRS3035RES200mN2805800E3777600,2021,3,1.696676,2.445000,3.0,2.0,0.748324,-0.748324,-0.748324,-0.748324
12,CRS3035RES200mN2805800E3777600,2022,11,7.320004,4.885002,2.0,2.0,-2.435001,2.435001,2.435001,2.435001
17,CRS3035RES200mN2805800E3777800,2018,9,14.645828,14.384995,2.0,3.0,-0.260834,0.260834,0.260834,0.260834


In [19]:
# >>> MODIFIED: monthly summary of Nighttime Cooling Deficit
night_metrics.groupby("month")["NCD"].agg(["mean", "std", "count"]).round(2)


,mean,std,count
month,,,
1,1.68,1.45,8300
3,-2.60,1.80,82741
5,-3.16,1.04,42410
7,-3.51,1.96,204445
9,-1.61,2.07,272541
11,1.02,2.66,127910


In [20]:
# >>> MODIFIED: Mean NCD by year for July (month == 7)
july_ncd = night_metrics[night_metrics["month"] == 7]

mean_ncd_by_year = (
    july_ncd
    .groupby("year")["NCD"]
    .agg(["mean", "std", "count"])
    .round(3)
)

print("Mean Nighttime Cooling Deficit (NCD) by year, July only:")
print(mean_ncd_by_year)


Mean Nighttime Cooling Deficit (NCD) by year, July only:
       mean    std  count
year                     
2019 -3.003  1.446  60730
2020 -3.296  1.497   1510
2021 -7.350  1.426   7675
2022 -4.929  1.805  48205
2023 -2.656  1.451  43987
2025 -2.800  1.606  42338


In [21]:
# >>> MODIFIED: retain July observations using the revised NCD outcome
ncd_july = night_metrics[night_metrics["month"] == 7].copy()

# Count distinct years per grid cell
years_per_grid = (
    ncd_july
    .groupby(GRID_ID_FIELD)["year"]
    .nunique()
    .reset_index(name="n_years")
)

# Summary distribution
years_per_grid["n_years"].value_counts().sort_index()


n_years
1    11799
2    14235
3    10297
4    23674
5     7467
6      209
Name: count, dtype: int64

In [22]:
# Share of grid cells represented in the July NCD dataset
ncd_july.shape[0] / grid.shape[0]


2.194958289941273

In [23]:
# >>> MODIFIED: variance decomposition for NCD
overall_var = ncd_july["NCD"].var()
overall_mean = ncd_july["NCD"].mean()

grid_means = (
    ncd_july
    .groupby(GRID_ID_FIELD)["NCD"]
    .mean()
)
between_grid_var = grid_means.var()

year_means = (
    ncd_july
    .groupby("year")["NCD"]
    .mean()
)
between_year_var = year_means.var()

within_grid_var = (
    ncd_july
    .groupby(GRID_ID_FIELD)["NCD"]
    .var()
    .mean()
)

print("Overall NCD variance:", overall_var)
print("Overall NCD mean:", overall_mean)
print("Between-grid NCD variance:", between_grid_var)
print("Between-year NCD variance:", between_year_var)
print("Average within-grid NCD variance:", within_grid_var)


Overall NCD variance: 3.8349636
Overall NCD mean: -3.5058525
Between-grid NCD variance: 1.2282245
Between-year NCD variance: 3.3643456
Average within-grid NCD variance: 4.1123796


In [24]:
ncd_july.head()


night_bin,idcar_200m,year,month,early_morning,late_night,n_early_morning,n_late_night,cooling_magnitude_raw,NCD_raw,NCD,NCD_winsorized
185,CRS3035RES200mN2806400E3787000,2019,7,14.997168,19.900002,2.0,2.0,4.902834,-4.902834,-4.902834,-4.902834
299,CRS3035RES200mN2806600E3786000,2019,7,14.903335,16.898895,3.0,2.0,1.995561,-1.995561,-1.995561,-1.995561
398,CRS3035RES200mN2806800E3786400,2019,7,13.869987,16.498011,3.0,3.0,2.628023,-2.628023,-2.628023,-2.628023
414,CRS3035RES200mN2806800E3786600,2019,7,14.529993,17.160858,2.0,2.0,2.630865,-2.630865,-2.630865,-2.630865
524,CRS3035RES200mN2807000E3786200,2019,7,14.838003,16.486668,3.0,2.0,1.648664,-1.648664,-1.648664,-1.648664


In [25]:
ncd_july.describe()


night_bin,year,month,early_morning,late_night,n_early_morning,n_late_night,cooling_magnitude_raw,NCD_raw,NCD,NCD_winsorized
count,204445.000000,204445.0,204445.000000,204445.000000,204445.000000,204445.000000,204445.000000,204445.000000,204445.000000,204445.000000
mean,2021.892959,7.0,16.156736,19.662588,3.578581,4.316188,3.505852,-3.505852,-3.505852,-3.453182
std,2.192721,0.0,1.766720,1.913755,1.805151,1.627391,1.958306,1.958306,1.958306,1.731959
min,2019.000000,7.0,9.434003,1.733337,2.000000,2.000000,-12.957890,-32.372025,-32.372025,-7.197543
25%,2019.000000,7.0,14.800379,18.344641,2.000000,3.000000,2.172328,-4.692575,-4.692575,-4.692575
50%,2022.000000,7.0,15.953331,19.457085,3.000000,4.000000,3.362883,-3.362883,-3.362883,-3.362883
75%,2023.000000,7.0,17.347651,20.820770,5.000000,5.000000,4.692575,-2.172328,-2.172328,-2.172328
max,2025.000000,7.0,23.301941,46.780769,8.000000,9.000000,32.372025,12.957890,12.957890,4.280369


In [26]:
# Add geometry to July NCD data and create a GeoDataFrame
ncd_july = ncd_july.merge(
    grid[[GRID_ID_FIELD, "geometry"]],
    on=GRID_ID_FIELD,
    how="left"
)

ncd_july = gpd.GeoDataFrame(
    ncd_july,
    geometry="geometry",
    crs=grid.crs
)


In [27]:
ncd_july.columns


Index(['idcar_200m', 'year', 'month', 'early_morning', 'late_night',
       'n_early_morning', 'n_late_night', 'cooling_magnitude_raw', 'NCD_raw',
       'NCD', 'NCD_winsorized', 'geometry'],
      dtype='str')

In [28]:
paris = gpd.read_file(
    r"C:\Users\kbonsu\Desktop\AICC Conference\ECOSTRESS_data_processing\paris_coeur_hypercenter.gpkg"
).to_crs(grid.crs)

paris_boundary = paris.dissolve()

# >>> MODIFIED: clip the revised July NCD dataset to the study area
ncd_july_paris = gpd.overlay(
    ncd_july,
    paris_boundary,
    how="intersection"
)


In [29]:
# >>> THREE-PART REVISION: retain all raw thermal outcomes and QC variables
ncd_july_paris = ncd_july_paris[
    [
        "idcar_200m",
        "year",
        "month",
        "late_night",
        "early_morning",
        "n_late_night",
        "n_early_morning",
        "cooling_magnitude_raw",
        "NCD_raw",
        "NCD",
        "NCD_winsorized",
        "geometry"
    ]
].copy()

# Final pre-export checks
assert np.allclose(
    ncd_july_paris["NCD_raw"],
    ncd_july_paris["early_morning"] - ncd_july_paris["late_night"],
    equal_nan=True
)
assert np.allclose(
    ncd_july_paris["NCD_raw"],
    -ncd_july_paris["cooling_magnitude_raw"],
    equal_nan=True
)
assert np.allclose(
    ncd_july_paris["NCD"],
    ncd_july_paris["NCD_raw"],
    equal_nan=True
)

print("Pre-export identities verified.")


Pre-export identities verified.


In [30]:
# >>> THREE-PART REVISION: export internally consistent thermal outcomes
PREPROCESS_OUTPUT = (
    "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/"
    "ecostress_paris_july_three_part_thermal_outcomes_200m.gpkg"
)

ncd_july_paris.to_file(
    PREPROCESS_OUTPUT,
    driver="GPKG"
)

print("✅ Three-part July thermal outcomes saved:")
print(PREPROCESS_OUTPUT)


✅ Three-part July thermal outcomes saved:
C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/output/ecostress_paris_july_three_part_thermal_outcomes_200m.gpkg
